In [1]:
import importlib.util
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import time

from data_processing import DataProcessor
from transfer import TransferModel
from slim_config import SlimConfig

In [2]:
spec = importlib.util.spec_from_file_location("pc_model", "../pc_linreg/model.py")
pc_model = importlib.util.module_from_spec(spec); spec.loader.exec_module(pc_model)

In [3]:
# short name -> file. The fold loop holds each of these out in turn.
DATASETS = {
    "K562": "../../data/replogle_nadig/K562_gwps_raw_bulk_01.h5ad",
    "RPE1": "../../data/replogle_nadig/rpe1_raw_bulk_01.h5ad",
    "HepG2": "../../data/replogle_nadig/nadig_hepg2_pseudobulk.h5ad",
    "Jurkat": "../../data/replogle_nadig/nadig_jurkat_pseudobulk.h5ad",
}

In [4]:
gene_names = pd.read_csv("../../data/controls/gene_names.csv")["gene_name"].tolist()
dp = DataProcessor(gene_names)
config = SlimConfig()
model = TransferModel(gene_names, '../../data/embeddings/gene_string_embeddings.v0.3.h5', config, transfer_c =200)
truth = pc_model.Model(gene_names)

for name, path in DATASETS.items():
    t = time.time()
    d = truth._load_dataset_effects(path, "log2")
    n_perts, n_genes = d["effects"].shape
    print(f"{name:7s} {n_perts:5d} perturbations x {n_genes:5d} measured genes  ({time.time() - t:.0f}s)")

K562     7420 perturbations x  7679 measured genes  (1s)
RPE1     1921 perturbations x  8259 measured genes  (0s)
HepG2    1494 perturbations x  9023 measured genes  (0s)
Jurkat   1739 perturbations x  8283 measured genes  (0s)


In [5]:
N_TOP = 50  # "top genes" per perturbation, standing in for "significantly changed"
METRICS = ["pds", "mse", "nmae", "jac", "fid", "dir"]
LOWER_IS_BETTER = {"pds", "mse", "nmae"}


def _top_idx(M, n):
    # row indices of the n largest |values| in each column of M -> shape (n, n_perts)
    return np.argpartition(-np.abs(M), n, axis=0)[:n]


PANEL_SIZE = 300  # the real challenge ranks each prediction among 300 perturbations


def pds_rank(pred, true, target_rows, seed=0):
    # For perturbation p: what fraction of the OTHER perturbations' real effects in its
    # panel are closer (cosine similarity) to pred_p than p's own real effect is?
    # 0 = perfect, 0.5 = random. Ties count half.
    #
    # Like the official PDS, perturbations are split into panels of ~300, and every
    # panel member's target gene is removed from all comparisons in that panel. The
    # fixed seed gives every predictor the same panels, so their scores are comparable.
    P = pred.shape[1]
    out = np.empty(P)
    order = np.random.default_rng(seed).permutation(P)
    for panel in np.array_split(order, max(1, round(P / PANEL_SIZE))):
        keep = np.ones(pred.shape[0], bool)
        rows = target_rows[panel]
        keep[rows[rows >= 0]] = False
        pp, tt = pred[keep][:, panel], true[keep][:, panel]
        pn = pp / (np.linalg.norm(pp, axis=0, keepdims=True) + 1e-12)
        tn = tt / (np.linalg.norm(tt, axis=0, keepdims=True) + 1e-12)
        sim = pn.T @ tn  # sim[i, j] = cosine(pred of panel[i], truth of panel[j])
        own = np.diag(sim)[:, None]
        closer = (sim > own).sum(axis=1) + 0.5 * ((sim == own).sum(axis=1) - 1)
        out[panel] = closer / (len(panel) - 1)
    return out


def per_pert_metrics(pred, true, target_rows, n_top=N_TOP):
    cols = np.arange(true.shape[1])
    true_top = _top_idx(true, n_top)  # genes that really changed most
    pred_top = _top_idx(pred, n_top)  # genes the prediction says changed most

    in_true = np.zeros(true.shape, bool)
    in_true[true_top, cols] = True
    in_pred = np.zeros(true.shape, bool)
    in_pred[pred_top, cols] = True
    overlap = (in_true & in_pred).sum(axis=0)

    return pd.DataFrame({
        "pds": pds_rank(pred, true, target_rows),
        "mse": ((pred - true) ** 2).mean(axis=0),
        "nmae": np.abs(pred[true_top, cols] - true[true_top, cols]).mean(axis=0),
        "jac": overlap / (2 * n_top - overlap),
        "fid": (np.sign(pred[pred_top, cols]) == np.sign(true[pred_top, cols])).mean(axis=0),
        "dir": (np.sign(pred[true_top, cols]) == np.sign(true[true_top, cols])).mean(axis=0),
    })

In [6]:
N_EVAL_GENES = 2000
EXCLUDE_FRAC = 0.25


def run_fold(model, held_out, K=10, lam=0.1, alpha=1.0, seed=0):
    # model: the SLIM Model being scored. Uses one global from the setup cell:
    #   dp -- the DataProcessor that builds SLIM's training matrix Y, and also
    #         supplies the held-out screen's truth in the SAME log1p-normalized
    #         space SLIM predicts in (pc_linreg's loader mixed raw counts and
    #         already-logged values, so its truth was on a different scale)
    # Expects model.predict_effects(targets, knockdown_fraction=None, gene_rows=None,
    # alpha=1.0) -> (len(gene_rows), len(targets)) log2FC, gene_rows = global VCC indices.

    # 1. split
    train_names = [n for n in DATASETS if n != held_out]
    held = dp.load_dataset(held_out)
    cols = [dp.gene_index[g] for g in held.var_names]  # global gene indices
    # log2FC vs the held-out screen's own control: (log1p(cp10k) - ctrl) / ln 2
    effects = pd.DataFrame((held.X - held.var["ctrl_mean"].to_numpy()) / np.log(2),
                           index=held.obs_names, columns=cols)
    ctrl = pd.Series(held.var["ctrl_mean"].to_numpy(), index=cols)
    held_targets = effects.index.to_numpy()

    # 2. genes to score
    eval_rows = ctrl.nlargest(N_EVAL_GENES).index.to_numpy()  # global gene indices

    # 3. covered vs. uncovered (+ hide some covered ones on purpose)
    trained_on = set().union(*(dp.load_dataset(n).obs_names for n in train_names))
    covered = np.isin(held_targets, list(trained_on))
    rng = np.random.default_rng(seed)
    hidden = rng.choice(held_targets[covered], size=int(EXCLUDE_FRAC * covered.sum()), replace=False)
    group = np.where(covered & ~np.isin(held_targets, hidden), "covered", "uncovered")

    # 4. fit on training screens only (hidden targets dropped from every screen)
    Y, genes, perts = dp.build_Y(train_names, exclude_perts=hidden, genes = "union")
    model.config.K = K
    model.config.lambda_reg = lam
    measured = {n: dp.load_dataset(n).var_names for n in train_names}
    model.fit(Y, genes, perts, measured)
    model.set_context(dp.load_dataset(held_out).var["ctrl_mean"], dp.source_ctrls(train_names), panel = held_targets)
    print(held_out, dict(zip(model.source_names, model.weights.round(2))))

    # 5. predict + score every predictor on the same perturbations and genes
    true = effects[eval_rows].to_numpy().T  # (genes, perturbations)
    n = true.shape[1]
    row_of = {g: i for i, g in enumerate(eval_rows)}

    # dp keeps targets outside the challenge gene panel too, so look them up with .get
    target_rows = np.array([row_of.get(dp.gene_index.get(t, -1), -1) for t in held_targets])
    has = np.where(target_rows >= 0)[0]

    def mask_targets(M):  # the official metrics never score a perturbation's own target
        M = M.copy()
        M[target_rows[has], has] = 0
        return M

    true = mask_targets(true)
    predictors = {
        # the model as you'd submit it (the target's own gene is masked, so no knockdown override)
        "model": model.predict_effects(held_targets, None, gene_rows=eval_rows, alpha=alpha),
        # training screens' mean effect (b), same for every target: alpha=0 drops G W P.T.
        # If "model" can't beat this, G/W isn't helping.
        "regular_slim": model.predict_effects(held_targets, None, gene_rows = eval_rows, alpha = alpha, transfer_c = None),
        "ref": model.predict_effects(held_targets, None, gene_rows=eval_rows, alpha=0.0, transfer_c = None),
        # zero effect everywhere: what the first (baseline) submission amounted to
        "no_change": np.zeros_like(true),
        # held-out screen's TRUE mean effect: the leaderboard's 0 point
        "base": np.repeat(true.mean(axis=1, keepdims=True), n, axis=1),
    }
    predictors = {who: mask_targets(M) for who, M in predictors.items()}

    # MSE-optimal gain: least-squares slope of truth on the submitted prediction,
    # g* = sum(P*T) / sum(P^2). gain multiplies this same quantity in write_submission,
    # so a gain well above g* overshoots on MSE/NMAE. "covered" is what the VCC panel mostly is.
    P = predictors["model"]
    g_star = {grp: float((P[:, group == grp] * true[:, group == grp]).sum()
                         / (P[:, group == grp] ** 2).sum())
              for grp in ("covered", "uncovered")}
    print(held_out, "g*", {k: round(v, 2) for k, v in g_star.items()})

    out = pd.concat([per_pert_metrics(M, true, target_rows).add_prefix(f"{who}_")
                     for who, M in predictors.items()], axis=1)
    out.insert(0, "group", group)
    out.insert(0, "target", held_targets)
    out.insert(0, "held_out", held_out)
    out["g_star"] = out["group"].map(g_star)  # one value per (fold, group)
    return out

In [7]:
PANEL_MIX = {"covered": 229 / 300, "uncovered": 71 / 300}


def summarize(per_pert, who="model"):
    # who = any predictor name from run_fold ("model", "ref", "no_change", ...)
    rows = []
    for (held_out, group), d in per_pert.groupby(["held_out", "group"]):
        row = {"held_out": held_out, "group": group, "n_perts": len(d)}
        for m in METRICS:
            model_v, base_v = d[f"{who}_{m}"].mean(), d[f"base_{m}"].mean()
            if m in LOWER_IS_BETTER:
                row[m] = 1 - model_v / base_v  # error: 1 - model/baseline
            else:
                row[m] = (model_v - base_v) / (1 - base_v)  # agreement: share of the gap to perfect closed
        rows.append(row)
    s = pd.DataFrame(rows)
    s["overall"] = s[METRICS].mean(axis=1)
    return s


def panel_score(summary):
    w = summary["group"].map(PANEL_MIX)
    cols = METRICS + ["overall"]
    weighted = summary[cols].mul(w, axis=0).groupby(summary["held_out"]).sum()
    return weighted.div(w.groupby(summary["held_out"]).sum(), axis=0)  # renormalise if a group is empty

In [8]:
t = time.time()
one = run_fold(model, "RPE1", K=10, lam=0.1)
display(summarize(one).round(3))
side_by_side = pd.concat({"model": panel_score(summarize(one)),
                          "ref (training mean)": panel_score(summarize(one, who="ref"))})
display(side_by_side.round(3))

  Fitted on 12250 perts, STRING coverage: 11749/12250
  direct transfer from ('K562',): 9184 targets have a measurement
RPE1 {'K562': np.float64(0.28), 'HepG2': np.float64(0.42), 'Jurkat': np.float64(0.3)}
RPE1 g* {'covered': 1.49, 'uncovered': 1.92}


,held_out,group,n_perts,pds,mse,nmae,jac,fid,dir,overall
0,RPE1,covered,1507,0.253,-0.197,-0.237,-0.188,-0.067,-0.574,-0.168
1,RPE1,uncovered,509,0.053,-0.228,-0.278,-0.202,-0.074,-0.636,-0.228


,,pds,mse,nmae,jac,fid,dir,overall
,held_out,,,,,,,
model,RPE1,0.206,-0.204,-0.246,-0.191,-0.068,-0.589,-0.182
ref (training mean),RPE1,0.000,-0.265,-0.298,-0.193,-0.058,-0.675,-0.248


In [9]:
# alpha varies innermost: it only changes predict_effects, so fit() keeps the
# existing G/W/b and each alpha costs just a prediction + scoring
GRID = [dict(K=20, lam=lam, alpha=alpha)
        for lam in (100, 1000, 10000)
        for alpha in (1.5, 2, 2.5, 3.0)]

rows = []
for held_out in DATASETS:  # outer loop = fold (see above)
    for cfg in GRID:
        t = time.time()
        pp = run_fold(model, held_out, **cfg)
        s = panel_score(summarize(pp)).reset_index()
        s["ref_overall"] = panel_score(summarize(pp, who="ref"))["overall"].to_numpy()
        s["slim_overall"] = panel_score(summarize(pp, who="regular_slim"))["overall"].to_numpy()
        rows.append(s.assign(**cfg))
        print(f"{held_out:7s} {cfg}  overall={s['overall'].iloc[0]:.3f}  ({time.time() - t:.0f}s)")
grid = pd.concat(rows, ignore_index=True)

  Fitted on 4488 perts, STRING coverage: 4480/4488
  direct transfer from ('K562',): 0 targets have a measurement
K562 {'RPE1': np.float64(0.21), 'HepG2': np.float64(0.29), 'Jurkat': np.float64(0.5)}
K562 g* {'covered': 0.27, 'uncovered': 0.07}
K562    {'K': 20, 'lam': 100, 'alpha': 1.5}  overall=-0.157  (10s)
  Fitted on 4488 perts, STRING coverage: 4480/4488
  direct transfer from ('K562',): 0 targets have a measurement
K562 {'RPE1': np.float64(0.21), 'HepG2': np.float64(0.29), 'Jurkat': np.float64(0.5)}
K562 g* {'covered': 0.23, 'uncovered': 0.05}
K562    {'K': 20, 'lam': 100, 'alpha': 2}  overall=-0.197  (10s)
  Fitted on 4488 perts, STRING coverage: 4480/4488
  direct transfer from ('K562',): 0 targets have a measurement
K562 {'RPE1': np.float64(0.21), 'HepG2': np.float64(0.29), 'Jurkat': np.float64(0.5)}
K562 g* {'covered': 0.19, 'uncovered': 0.04}
K562    {'K': 20, 'lam': 100, 'alpha': 2.5}  overall=-0.257  (10s)
  Fitted on 4488 perts, STRING coverage: 4480/4488
  direct transf

In [12]:
REALISTIC = ["RPE1", "HepG2", "Jurkat"]

cfg_cols = list(GRID[0])
realistic = grid[grid["held_out"].isin(REALISTIC)]
table = realistic.groupby(cfg_cols)[METRICS + ["overall"]].mean()
table["realistic_sd"] = realistic.groupby(cfg_cols)["overall"].std()
table["ref_overall"] = realistic.groupby(cfg_cols)["ref_overall"].mean()  # to beat
table["slim_overall"] = realistic.groupby(cfg_cols)["slim_overall"].mean()
table["k562_fold"] = grid[grid["held_out"] == "K562"].set_index(cfg_cols)["overall"]
table.sort_values("overall", ascending=False).round(3)

pds    mse   nmae    jac    fid    dir  overall  \
K  lam   alpha                                                      
20 100   2.0    0.219 -0.086 -0.065 -0.086 -0.083 -0.475   -0.096   
         1.5    0.212 -0.086 -0.074 -0.085 -0.074 -0.472   -0.096   
         2.5    0.226 -0.092 -0.056 -0.087 -0.093 -0.484   -0.098   
   1000  3.0    0.203 -0.088 -0.066 -0.085 -0.080 -0.476   -0.099   
         2.5    0.200 -0.087 -0.072 -0.084 -0.075 -0.474   -0.099   
         2.0    0.197 -0.089 -0.079 -0.084 -0.073 -0.473   -0.100   
   100   3.0    0.233 -0.102 -0.048 -0.089 -0.106 -0.495   -0.101   
   1000  1.5    0.194 -0.092 -0.085 -0.084 -0.068 -0.474   -0.102   
   10000 3.0    0.179 -0.103 -0.097 -0.085 -0.072 -0.481   -0.110   
         2.5    0.179 -0.104 -0.098 -0.085 -0.071 -0.482   -0.110   
         2.0    0.179 -0.105 -0.100 -0.085 -0.071 -0.483   -0.111   
         1.5    0.179 -0.107 -0.101 -0.086 -0.071 -0.484   -0.112   

                realistic_sd  ref_overall  slim_overall  k562_fold  
K  lam   alpha                                                      
20 100   2.0           0.080       -0.186        -0.155     -0.197  
         1.5           0.078       -0.186        -0.154     -0.157  
         2.5           0.083       -0.186        -0.161     -0.257  
   1000  3.0           0.074       -0.186        -0.158     -0.167  
         2.5           0.074       -0.186        -0.158     -0.156  
         2.0           0.074       -0.186        -0.159     -0.152  
   100   3.0           0.086       -0.186        -0.169     -0.330  
   1000  1.5           0.073       -0.186        -0.163     -0.154  
   10000 3.0           0.071       -0.186        -0.179     -0.191  
         2.5           0.071       -0.186        -0.180     -0.192  
         2.0           0.072       -0.186        -0.181     -0.192  
         1.5           0.072       -0.186        -0.182     -0.193

In [11]:
# MSE-optimal gain at the submitted settings, one value per realistic fold.
# Compare against the gain you'd submit: well above g* = paying MSE/NMAE for FID.
g = pd.concat([run_fold(model, h, K=20, lam=100, alpha=1.5) for h in REALISTIC])
g.groupby(["held_out", "group"])["g_star"].first().unstack().round(2)

  Fitted on 12250 perts, STRING coverage: 11749/12250
  direct transfer from ('K562',): 9184 targets have a measurement
RPE1 {'K562': np.float64(0.28), 'HepG2': np.float64(0.42), 'Jurkat': np.float64(0.3)}
RPE1 g* {'covered': 1.44, 'uncovered': 1.71}
  Fitted on 12566 perts, STRING coverage: 12061/12566
  direct transfer from ('K562',): 9236 targets have a measurement
HepG2 {'K562': np.float64(0.37), 'RPE1': np.float64(0.37), 'Jurkat': np.float64(0.26)}
HepG2 g* {'covered': 0.87, 'uncovered': 0.98}
  Fitted on 12092 perts, STRING coverage: 11586/12092
  direct transfer from ('K562',): 9157 targets have a measurement
Jurkat {'K562': np.float64(0.54), 'RPE1': np.float64(0.24), 'HepG2': np.float64(0.22)}
Jurkat g* {'covered': 0.67, 'uncovered': 0.67}


group,covered,uncovered
held_out,,
HepG2,0.87,0.98
Jurkat,0.67,0.67
RPE1,1.44,1.71
